In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 1: INSTALLS + IMPORTS
# ════════════════════════════════════════════════════════════════════════

!pip install -q imbalanced-learn albumentations umap-learn

import os, warnings, json, gc
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, classification_report,
                              confusion_matrix, f1_score)
from sklearn.preprocessing import label_binarize, LabelEncoder
import albumentations as A

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

# GPU setup
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Mixed precision disabled — EfficientNetB4's internal preprocessing
    # does float32 ops that conflict with float16 input tensors
    print(f"✅ GPU: {gpus[0].name} | Mixed precision: OFF (EfficientNetB4 compat)")
else:
    print("⚠️  No GPU found — training will be slow")

print(f"✅ TensorFlow: {tf.__version__}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIG  ← UPDATE THESE PATHS TO MATCH YOUR SETUP
# ════════════════════════════════════════════════════════════════════════

# ── PATHS — change these to match YOUR file locations ────────────────
CURRENT_CSV = '/workspace/notebooks/preprocessed_validated_data.csv'
IMAGES_DIR  = 'ISIC-images'     # folder with all .jpg images
OUTPUT_DIR  = '/workspace/notebooks/sota_v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model config ──────────────────────────────────────────────────────
IMAGE_SIZE   = (380, 380)   # EfficientNetB4 native — much better than 300×300
BATCH_SIZE   = 24           # Reduced to fit 380×380 in GPU memory
NUM_CLASSES  = 3
CLASS_NAMES  = ['Melanoma', 'Benign Nevi', 'Other Malignant']
COLORS       = ['#E74C3C', '#2ECC71', '#3498DB']
MIXUP_ALPHA  = 0.2          # Kept low so training acc doesn't lag val acc

print(f"✅ Config ready")
print(f"   IMAGE_SIZE  : {IMAGE_SIZE}")
print(f"   BATCH_SIZE  : {BATCH_SIZE}")
print(f"   OUTPUT_DIR  : {OUTPUT_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 3: LOAD CSV + INSPECT
# ════════════════════════════════════════════════════════════════════════

df_raw = pd.read_csv(CURRENT_CSV)

print("=" * 60)
print(f"DATASET OVERVIEW  —  {len(df_raw):,} rows")
print("=" * 60)
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\n--- diagnosis_encoded distribution ---")
print(df_raw['diagnosis_encoded'].value_counts())
print(f"\n--- specific_label distribution ---")
print(df_raw['specific_label'].value_counts().head(15))
print(f"\n--- diagnosis_confirm_type ---")
print(df_raw['diagnosis_confirm_type'].value_counts())
print(f"\nNull counts (key columns):")
key_cols = ['diagnosis_encoded', 'age_approx', 'sex',
            'anatom_site_general', 'diagnosis_confirm_type']
print(df_raw[[c for c in key_cols if c in df_raw.columns]].isnull().sum())


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 4: LABEL MAPPING → 3 CLASSES
# ════════════════════════════════════════════════════════════════════════
# Your diagnosis_encoded values (from notebook inspection):
#   3 = Melanoma       → Class 0  (Melanoma)
#   4 = Nevus          → Class 1  (Benign)
#   1 = BKL            → Class 1  (Benign — seborrheic keratosis)
#   2 = DF             → Class 1  (Benign — dermatofibroma)
#   0 = BCC            → Class 2  (Other Malignant)
#   5 = SCC            → Class 2  (Other Malignant)
#   6 = Unknown        → EXCLUDE

df = df_raw.copy()

label_remap = {
    3: 0,    # Melanoma
    4: 1,    # Nevus → Benign
    1: 1,    # BKL → Benign
    2: 1,    # DF → Benign
    0: 2,    # BCC → Other Malignant
    5: 2,    # SCC → Other Malignant
    6: -1,   # Unknown → exclude
}

df['label'] = df['diagnosis_encoded'].map(label_remap)

n_before = len(df)
df = df[df['label'] != -1].reset_index(drop=True)
print(f"Dropped {n_before - len(df):,} Unknown rows")

print(f"\n✅ 3-CLASS DISTRIBUTION:")
for i, name in enumerate(CLASS_NAMES):
    count = (df['label'] == i).sum()
    print(f"  {i} = {name}: {count:,} ({count/len(df)*100:.1f}%)")



In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 5: FEATURE ENGINEERING (METADATA)
# ════════════════════════════════════════════════════════════════════════

# ── Fill nulls ────────────────────────────────────────────────────────
df['sex']                 = df['sex'].fillna('unknown')
df['anatom_site_general'] = df['anatom_site_general'].fillna('unspecified')
df['age_approx']          = df['age_approx'].fillna(df['age_approx'].median())

# ── Sex encoding ──────────────────────────────────────────────────────
df['sex_enc'] = df['sex'].map({'male': 0.0, 'female': 1.0, 'unknown': 0.5})

# ── Age features (continuous + bin flags) ─────────────────────────────
df['age_norm']  = df['age_approx'] / 100.0
df['age_lt35']  = (df['age_approx'] < 35).astype(float)
df['age_35_50'] = ((df['age_approx'] >= 35) & (df['age_approx'] < 50)).astype(float)
df['age_50_65'] = ((df['age_approx'] >= 50) & (df['age_approx'] < 65)).astype(float)
df['age_gt65']  = (df['age_approx'] >= 65).astype(float)

# ── Anatomic site: one-hot encoding ───────────────────────────────────
site_dummies = pd.get_dummies(
    df['anatom_site_general'], prefix='site', dummy_na=False
).astype(float)
# Add any dummy columns that might be missing in a subset
df = pd.concat([df, site_dummies], axis=1)
new_site_cols = [c for c in site_dummies.columns]

# ── Lesion size (if available) ─────────────────────────────────────────
if 'clin_size_long_diam_mm' in df.columns:
    df['size_norm'] = (df['clin_size_long_diam_mm']
                       .fillna(df['clin_size_long_diam_mm'].median()) / 50.0)
    size_col = ['size_norm']
    print(f"✅ Lesion size feature added")
else:
    size_col = []

# ── Fitzpatrick skin type (if available) ──────────────────────────────
if 'fitzpatrick_skin_type' in df.columns:
    fitz_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6,
                '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6,
                  1: 1,   2: 2,   3: 3,   4: 4,   5: 5,   6: 6}
    df['skin_type_norm'] = df['fitzpatrick_skin_type'].map(fitz_map).fillna(3) / 6.0
    skin_col = ['skin_type_norm']
    print(f"✅ Fitzpatrick skin type added")
else:
    skin_col = []

# ── Melanocytic flag (strong discriminative signal) ───────────────────
if 'melanocytic' in df.columns:
    df['melanocytic_flag'] = df['melanocytic'].map(
        {True: 1.0, False: 0.0, 'true': 1.0, 'false': 0.0,
         1: 1.0, 0: 0.0}
    ).fillna(0.5)
    melan_col = ['melanocytic_flag']
    print(f"✅ Melanocytic flag added")
else:
    melan_col = []

# ── Remove duplicate columns first (CSV had site_ cols AND we made new ones) ──
df = df.loc[:, ~df.columns.duplicated()].copy()

# ── Use CSV's pre-encoded cols if they exist ──────────────────────────────────
existing_site_from_csv = [c for c in df_raw.columns if c.startswith('site_')]
final_site_cols = existing_site_from_csv if existing_site_from_csv else new_site_cols
age_norm_col    = ['age_normalized'] if 'age_normalized' in df.columns else ['age_norm']
sex_col_meta    = ['sex_encoded']    if 'sex_encoded'    in df.columns else ['sex_enc']

raw_meta = (sex_col_meta + age_norm_col
            + ['age_lt35', 'age_35_50', 'age_50_65', 'age_gt65']
            + final_site_cols + size_col + skin_col + melan_col)

seen = set()
META_COLS = []
for col in raw_meta:
    if col not in seen and col in df.columns:
        META_COLS.append(col)
        seen.add(col)

META_DIM = len(META_COLS)

# Safe to run now — no duplicate columns
for col in META_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f"\n✅ Metadata features: {META_DIM} total")
print(f"   {META_COLS}")


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 6: IMAGE PATH FILTER + TRAIN/VAL SPLIT
# ════════════════════════════════════════════════════════════════════════

# Build image_path column if not already there
if 'image_path' not in df.columns:
    # Try common naming conventions
    if 'isic_id' in df.columns:
        df['image_path'] = df['isic_id'].apply(
            lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg")
        )
    elif 'image_name' in df.columns:
        df['image_path'] = df['image_name'].apply(
            lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg")
        )
    else:
        raise ValueError("Cannot find image ID column. Check your CSV columns.")
    print("✅ Built image_path column")
else:
    print("✅ image_path column already exists")

# Filter to images that actually exist on disk
print("\n🔍 Checking images on disk (may take ~30 seconds)...")
df['img_exists'] = df['image_path'].apply(os.path.exists)
found   = df['img_exists'].sum()
missing = (~df['img_exists']).sum()
print(f"   Found on disk : {found:,}")
print(f"   Missing       : {missing:,} ({missing/len(df)*100:.1f}%)")

df = df[df['img_exists']].drop(columns=['img_exists']).reset_index(drop=True)
print(f"\n📦 Clean dataset: {len(df):,} images")
print(f"Class dist:")
for i, name in enumerate(CLASS_NAMES):
    count = (df['label'] == i).sum()
    print(f"  {name}: {count:,}")

# Train / val split (stratified)
train_df, val_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,}")
print(f"Train dist: {train_df['label'].value_counts().sort_index().to_dict()}")
print(f"Val dist:   {val_df['label'].value_counts().sort_index().to_dict()}")



In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 7: AUGMENTATION PIPELINE
# ════════════════════════════════════════════════════════════════════════

# ── Dermoscopy-specific artifact functions ────────────────────────────
def add_hair(img, num_hairs=8):
    """Synthetic hair artifacts present in real dermoscopy images"""
    img = img.copy()
    h, w = img.shape[:2]
    for _ in range(num_hairs):
        x1        = np.random.randint(0, w)
        y1        = np.random.randint(0, h)
        angle     = np.random.uniform(0, np.pi)
        length    = np.random.randint(40, 140)
        thickness = np.random.randint(1, 3)
        x2        = int(x1 + length * np.cos(angle))
        y2        = int(y1 + length * np.sin(angle))
        darkness  = np.random.randint(0, 60)
        cv2.line(img, (x1, y1), (x2, y2),
                 (darkness, darkness, darkness),
                 thickness, lineType=cv2.LINE_AA)
    return img

def add_dark_corner(img):
    """Dermoscopy vignette — dark circular border"""
    h, w   = img.shape[:2]
    cx, cy = w // 2, h // 2
    Y, X   = np.ogrid[:h, :w]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2)
    max_d  = np.sqrt(cx**2 + cy**2)
    mask   = 1.0 - np.clip((dist / max_d - 0.7) / 0.3, 0, 1)
    return (img * mask[:, :, np.newaxis]).astype(np.uint8)

# ── Albumentations pipeline ───────────────────────────────────────────
# NOTE: NO A.Normalize — EfficientNet normalizes internally. Feed [0,255].
aug_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                       rotate_limit=40, border_mode=0, p=0.55),
    A.OneOf([
        A.RandomBrightnessContrast(0.25, 0.25, p=1.0),
        A.HueSaturationValue(15, 25, 15, p=1.0),
        A.CLAHE(clip_limit=3.0, p=1.0),
    ], p=0.70),
    A.OneOf([
        A.GaussNoise(p=1.0),
        A.GaussianBlur(blur_limit=3, p=1.0),
        A.MedianBlur(blur_limit=3, p=1.0),
    ], p=0.25),
    A.OneOf([
        A.ElasticTransform(alpha=60, sigma=6, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
    ], p=0.20),
    A.CoarseDropout(max_holes=6, max_height=28, max_width=28,
                    fill_value=0, p=0.35),
])

def apply_aug_train(img):
    if np.random.random() < 0.45:
        img = add_hair(img, num_hairs=np.random.randint(3, 10))
    if np.random.random() < 0.15:
        img = add_dark_corner(img)
    return aug_train(image=img)['image'].astype(np.float32)

def apply_aug_val(img):
    return img.astype(np.float32)

# ── 8-pass TTA (used at inference time only) ──────────────────────────
TTA_AUGS = [
    A.Compose([]),
    A.Compose([A.HorizontalFlip(p=1.0)]),
    A.Compose([A.VerticalFlip(p=1.0)]),
    A.Compose([A.Transpose(p=1.0)]),
    A.Compose([A.RandomRotate90(p=1.0)]),
    A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)]),
    A.Compose([A.Transpose(p=1.0), A.HorizontalFlip(p=1.0)]),
    A.Compose([A.RandomRotate90(p=1.0), A.HorizontalFlip(p=1.0)]),
]

print(f"✅ Augmentation ready")
print(f"   Input format : float32 [0,255] — EfficientNet normalizes internally")
print(f"   Hair aug     : 45% chance/image")
print(f"   Vignette     : 15% chance/image")
print(f"   TTA passes   : {len(TTA_AUGS)}")


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 8: tf.data PIPELINE (MixUp + CutMix)
# ════════════════════════════════════════════════════════════════════════

def load_and_preprocess(path, meta, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.uint8)
    if augment:
        img = tf.numpy_function(apply_aug_train, [img], tf.float32)
    else:
        img = tf.numpy_function(apply_aug_val,   [img], tf.float32)
    img = tf.ensure_shape(img, [*IMAGE_SIZE, 3])
    return (
        {'imageinput':    img,
         'metadatainput': tf.cast(meta, tf.float32)},
        tf.one_hot(tf.cast(label, tf.int32), NUM_CLASSES)
    )

@tf.function
def mixup_batch(inputs, labels):
    """MixUp: blend two images with a random ratio"""
    imgs, metas = inputs['imageinput'], inputs['metadatainput']
    bsz = tf.shape(imgs)[0]
    lam = tf.random.uniform([], MIXUP_ALPHA, 1.0 - MIXUP_ALPHA)
    idx = tf.random.shuffle(tf.range(bsz))
    return (
        {'imageinput':    lam * imgs  + (1-lam) * tf.gather(imgs,  idx),
         'metadatainput': lam * metas + (1-lam) * tf.gather(metas, idx)},
        lam * labels + (1-lam) * tf.gather(labels, idx)
    )

@tf.function
def cutmix_batch(inputs, labels):
    """CutMix: paste a rectangular patch from another image"""
    imgs, metas = inputs['imageinput'], inputs['metadatainput']
    bsz = tf.shape(imgs)[0]
    H   = tf.cast(IMAGE_SIZE[0], tf.float32)
    W   = tf.cast(IMAGE_SIZE[1], tf.float32)

    lam   = tf.random.uniform([], MIXUP_ALPHA, 1.0 - MIXUP_ALPHA)
    cut_h = tf.cast(H * tf.sqrt(1.0 - lam), tf.int32)
    cut_w = tf.cast(W * tf.sqrt(1.0 - lam), tf.int32)

    cx = tf.random.uniform([], 0, IMAGE_SIZE[1], dtype=tf.int32)
    cy = tf.random.uniform([], 0, IMAGE_SIZE[0], dtype=tf.int32)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, IMAGE_SIZE[1])
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, IMAGE_SIZE[0])
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, IMAGE_SIZE[1])
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, IMAGE_SIZE[0])

    actual_lam = 1.0 - tf.cast((y2-y1)*(x2-x1), tf.float32) / (H * W)

    idx     = tf.random.shuffle(tf.range(bsz))
    imgs_b  = tf.gather(imgs, idx)

    rows = tf.range(IMAGE_SIZE[0])
    cols = tf.range(IMAGE_SIZE[1])
    row_mask = tf.cast((rows >= y1) & (rows < y2), tf.float32)[:, tf.newaxis, tf.newaxis]
    col_mask = tf.cast((cols >= x1) & (cols < x2), tf.float32)[tf.newaxis, :, tf.newaxis]
    patch_mask = row_mask * col_mask
    keep_mask  = 1.0 - patch_mask

    mixed_imgs   = imgs * keep_mask + imgs_b * patch_mask
    mixed_labels = actual_lam * labels + (1-actual_lam) * tf.gather(labels, idx)
    mixed_metas  = actual_lam * metas  + (1-actual_lam) * tf.gather(metas,  idx)
    return {'imageinput': mixed_imgs, 'metadatainput': mixed_metas}, mixed_labels

@tf.function
def random_augment_batch(inputs, labels):
    """Randomly choose MixUp or CutMix each batch"""
    return tf.cond(
        tf.random.uniform([]) > 0.5,
        lambda: cutmix_batch(inputs, labels),
        lambda: mixup_batch(inputs, labels)
    )

def build_dataset(df, meta_cols, augment=False, shuffle=True, mix=False):
    paths  = df['image_path'].values
    metas  = df[meta_cols].values.astype(np.float32)
    labels = df['label'].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, metas, labels))
    if shuffle:
        ds = ds.shuffle(len(df), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, m, l: load_and_preprocess(p, m, l, augment=augment),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE)
    if mix:
        ds = ds.map(random_augment_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = build_dataset(train_df, META_COLS, augment=True,  shuffle=True,  mix=True)
val_ds   = build_dataset(val_df,   META_COLS, augment=False, shuffle=False, mix=False)

# Sanity check
for (inputs, _) in train_ds.take(1):
    mn = inputs['imageinput'].numpy().min()
    mx = inputs['imageinput'].numpy().max()
    print(f"✅ Image batch  : {inputs['imageinput'].shape}")
    print(f"✅ Meta batch   : {inputs['metadatainput'].shape}")
    print(f"✅ Image range  : [{mn:.1f}, {mx:.1f}]  — must be 0–255 for EfficientNet")
    if mx < 2.0:
        raise RuntimeError("🚨 Images are already normalized to [0,1]! "
                           "Remove any A.Normalize from aug_train.")


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 9: MODEL — EfficientNetB4 + GeM + CBAM + FrozenBN
# ════════════════════════════════════════════════════════════════════════

class GeM(tf.keras.layers.Layer):
    """
    Generalized Mean Pooling (Radenović et al. 2018).
    Trainable p sits between average (p=1) and max pooling (p→∞).
    p=3 is empirically best for fine-grained recognition tasks.
    Proven +1–2% over GAP on image retrieval/classification benchmarks.
    """
    def __init__(self, p_init=3.0, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps    = eps
        self.p_init = p_init

    def build(self, input_shape):
        self.p = self.add_weight(
            name='gem_p',
            shape=(),
            initializer=tf.constant_initializer(self.p_init),
            trainable=True,
            constraint=tf.keras.constraints.MinMaxNorm(
                min_value=1.0, max_value=6.0, rate=1.0, axis=None
            )
        )

    def call(self, x):
        x   = tf.clip_by_value(x, self.eps, tf.float32.max)
        x_p = tf.pow(x, self.p)
        avg = tf.reduce_mean(x_p, axis=[1, 2])
        return tf.pow(avg, 1.0 / self.p)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'p_init': self.p_init, 'eps': self.eps})
        return cfg


class FrozenBNBackbone(tf.keras.layers.Layer):
    """
    Wraps backbone so BatchNorm always runs in inference mode.
    Critical: BN layers keep ImageNet running stats even during fine-tuning.
    Gradients still flow through weights — only BN statistics are protected.
    """
    def __init__(self, base_model, **kwargs):
        super().__init__(**kwargs)
        self.base_model = base_model

    def call(self, inputs, training=None):
        return self.base_model(inputs, training=False)

    def get_config(self):
        return super().get_config()


def build_model(img_shape=(*IMAGE_SIZE, 3), meta_dim=META_DIM):

    # ── BACKBONE: EfficientNetB4 @ 380×380 ───────────────────────────
    img_input = tf.keras.Input(shape=img_shape, name='imageinput')

    _base = tf.keras.applications.EfficientNetB4(
        input_shape=img_shape,
        weights='imagenet',
        include_top=False,
    )
    _base.trainable = False

    frozen_base = FrozenBNBackbone(_base, name='efficientnetb4_fbn')
    x = frozen_base(img_input)   # (batch, 12, 12, 1792)

    # ── CBAM CHANNEL ATTENTION ────────────────────────────────────────
    filters  = x.shape[-1]       # 1792
    gap_cbam = tf.keras.layers.GlobalAveragePooling2D()(x)
    gmp_cbam = tf.keras.layers.GlobalMaxPooling2D()(x)
    shared1  = tf.keras.layers.Dense(filters // 16, activation='swish', use_bias=False)
    shared2  = tf.keras.layers.Dense(filters, activation='sigmoid', use_bias=False)
    ch_att   = tf.keras.layers.Add()([shared2(shared1(gap_cbam)),
                                      shared2(shared1(gmp_cbam))])
    ch_att   = tf.keras.layers.Reshape((1, 1, filters))(ch_att)
    x        = tf.keras.layers.Multiply()([x, ch_att])

    # ── GeM POOLING (replaces standard GAP) ──────────────────────────
    img_feat = GeM(p_init=3.0, name='gem_pool')(x)
    img_feat = tf.keras.layers.BatchNormalization()(img_feat)
    img_feat = tf.keras.layers.Dropout(0.35)(img_feat)

    # ── METADATA MLP ──────────────────────────────────────────────────
    meta_input = tf.keras.Input(shape=(meta_dim,), name='metadatainput')
    m = tf.keras.layers.Dense(256, activation='swish')(meta_input)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Dropout(0.3)(m)
    m = tf.keras.layers.Dense(128, activation='swish')(m)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Dropout(0.2)(m)
    meta_feat = tf.keras.layers.Dense(64, activation='swish')(m)

    # ── FUSION + CLASSIFICATION HEAD ──────────────────────────────────
    fused = tf.keras.layers.Concatenate()([img_feat, meta_feat])
    fused = tf.keras.layers.Dense(768, activation='swish')(fused)
    fused = tf.keras.layers.BatchNormalization()(fused)
    fused = tf.keras.layers.Dropout(0.40)(fused)
    fused = tf.keras.layers.Dense(384, activation='swish')(fused)
    fused = tf.keras.layers.BatchNormalization()(fused)
    fused = tf.keras.layers.Dropout(0.30)(fused)
    fused = tf.keras.layers.Dense(128, activation='swish')(fused)
    fused = tf.keras.layers.Dropout(0.20)(fused)

    # dtype='float32' ensures output is full precision even with mixed_float16
    out = tf.keras.layers.Dense(
        NUM_CLASSES, activation='softmax',
        dtype='float32', name='output'
    )(fused)

    return tf.keras.Model([img_input, meta_input], out), _base


model, backbone = build_model()
print(f"✅ EfficientNetB4 + GeM + CBAM + FrozenBN Wrapper")
print(f"   Total params  : {model.count_params()/1e6:.1f}M")
print(f"   Input res     : {IMAGE_SIZE}")
print(f"   Pooling       : GeM (trainable p, better than GAP)")
print(f"   Pi5 TFLite    : ~23MB INT8 ✅")




In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 10: LOSS + HELPERS
# ════════════════════════════════════════════════════════════════════════

class AsymmetricLoss(tf.keras.losses.Loss):
    """
    Asymmetric Loss (ASL) — Ridnik et al. 2020 (arxiv:2009.14119)
    Uses DIFFERENT focal exponents for positives vs negatives:
      gamma_neg=4 aggressively ignores easy negatives (majority class)
      gamma_pos=1 stays gentle on true positive examples
    Consistently outperforms standard Focal Loss on imbalanced medical data.
    clip=0.05 shifts negative probability floor (prevents trivial solutions).
    """
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0,
                 clip=0.05, label_smooth=0.10, **kwargs):
        super().__init__(**kwargs)
        self.gamma_neg    = gamma_neg
        self.gamma_pos    = gamma_pos
        self.clip         = clip
        self.label_smooth = label_smooth

    def call(self, y_true, y_pred):
        # Label smoothing
        n_cls = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_s   = y_true * (1 - self.label_smooth) + self.label_smooth / n_cls

        y_pred     = tf.clip_by_value(y_pred, 1e-8, 1.0 - 1e-8)
        y_pred_neg = tf.maximum(y_pred - self.clip, 0.0)

        log_pos = -y_s     * tf.math.log(y_pred)
        log_neg = -(1-y_s) * tf.math.log(1 - y_pred_neg)

        pt_pos = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        pt_neg = 1.0 - y_pred

        loss = (tf.pow(1 - pt_pos, self.gamma_pos) * log_pos +
                tf.pow(pt_neg,     self.gamma_neg) * log_neg)
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma_neg': self.gamma_neg, 'gamma_pos': self.gamma_pos,
                    'clip': self.clip, 'label_smooth': self.label_smooth})
        return cfg


class SWACallback(tf.keras.callbacks.Callback):
    """Stochastic Weight Averaging — free +0.5–1% AUC by averaging checkpoints"""
    def __init__(self, start_epoch=3, save_path=None):
        super().__init__()
        self.start_epoch = start_epoch
        self.save_path   = save_path
        self.swa_weights = None
        self.n_averaged  = 0

    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            w = self.model.get_weights()
            if self.swa_weights is None:
                self.swa_weights = [np.zeros_like(wi) for wi in w]
            for i, wi in enumerate(w):
                self.swa_weights[i] += wi
            self.n_averaged += 1

    def on_train_end(self, logs=None):
        if self.swa_weights and self.n_averaged > 0:
            avg_w = [wi / self.n_averaged for wi in self.swa_weights]
            self.model.set_weights(avg_w)
            print(f"\n✅ SWA: averaged {self.n_averaged} checkpoints")
            if self.save_path:
                self.model.save(self.save_path)
                print(f"   Saved → {self.save_path}")


def warmup_cosine(epoch, total_epochs, warmup_epochs, peak_lr, min_lr=1e-7):
    """Linear warmup → cosine decay LR schedule"""
    if epoch < warmup_epochs:
        return float(peak_lr * (epoch + 1) / max(warmup_epochs, 1))
    progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
    return float(min_lr + 0.5 * (peak_lr - min_lr) * (1 + np.cos(np.pi * progress)))


def set_unfreeze(backbone, pct):
    """Unfreeze top pct% of backbone layers, BN always stays frozen"""
    total       = len(backbone.layers)
    keep_frozen = int(total * (1.0 - pct))
    n_bn_frozen = 0
    for i, layer in enumerate(backbone.layers):
        if i < keep_frozen:
            layer.trainable = False
        else:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
                n_bn_frozen += 1
            else:
                layer.trainable = True
    n_train = sum(1 for l in backbone.layers if l.trainable)
    print(f"   Backbone: {n_train}/{total} trainable | {n_bn_frozen} BN layers frozen ✅")


def compile_model(model, lr, wd=1e-4):
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            lr, weight_decay=wd, clipnorm=1.0
        ),
        loss=AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0,
                            clip=0.05, label_smooth=0.10),
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
        ]
    )


# ── Class weights: boosted for minority/critical classes ─────────────
counts_train = train_df['label'].value_counts().sort_index()
N            = len(train_df)
class_weights_tuned = {
    0: round(N / (NUM_CLASSES * counts_train[0]) * 2.5, 3),  # Melanoma ×2.5
    1: round(N / (NUM_CLASSES * counts_train[1]),       3),  # Benign baseline
    2: round(N / (NUM_CLASSES * counts_train[2]) * 1.5, 3),  # OtherMal ×1.5
}
print(f"✅ Loss, helpers, class weights ready")
print(f"⚖️  Class weights: {class_weights_tuned}")
print(f"   Mel={class_weights_tuned[0]:.2f}× | "
      f"Ben={class_weights_tuned[1]:.2f}× | "
      f"OthMal={class_weights_tuned[2]:.2f}×")


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 11: AUTOLOOP TRAINING
# ════════════════════════════════════════════════════════════════════════

TARGET_AUC       = 0.90
TARGET_ACC       = 0.85
MAX_TOTAL_EPOCHS = 350

PHASES = [
    {'pct': 0.00, 'lr': 8e-4, 'epochs': 35, 'warmup': 10, 'wd': 1e-4,
     'name': 'Ph1 Head only (backbone frozen)',   'swa': False},
    {'pct': 0.50, 'lr': 3e-5, 'epochs': 45, 'warmup': 5,  'wd': 5e-5,
     'name': 'Ph2 Top 50% unfrozen',              'swa': False},
    {'pct': 1.00, 'lr': 1e-5, 'epochs': 45, 'warmup': 3,  'wd': 2e-5,
     'name': 'Ph3 Full unfreeze',                 'swa': False},
    {'pct': 1.00, 'lr': 4e-6, 'epochs': 40, 'warmup': 0,  'wd': 1e-5,
     'name': 'Ph4 Squeeze + SWA',                 'swa': True},
    {'pct': 1.00, 'lr': 1e-6, 'epochs': 30, 'warmup': 0,  'wd': 5e-6,
     'name': 'Ph5 Deep squeeze + SWA',            'swa': True},
    {'pct': 1.00, 'lr': 3e-7, 'epochs': 25, 'warmup': 0,  'wd': 1e-6,
     'name': 'Ph6 Final polish + SWA',            'swa': True},
]

# ── Fresh start ───────────────────────────────────────────────────────
tf.keras.backend.clear_session()
gc.collect()
model, backbone = build_model()

best_val_auc     = 0.0
best_val_acc     = 0.0
total_epochs_run = 0
targets_met      = False
all_phase_log    = []

print("\n" + "=" * 65)
print(f"🎯  TARGETS: val_auc ≥ {TARGET_AUC}  AND  val_acc ≥ {TARGET_ACC}")
print(f"    Hard stop at {MAX_TOTAL_EPOCHS} epochs")
print(f"    Backbone: EfficientNetB4 @ {IMAGE_SIZE}")
print(f"    Loss: Asymmetric Loss (ASL)")
print("=" * 65)

for ph_i, phase in enumerate(PHASES):
    if targets_met or total_epochs_run >= MAX_TOTAL_EPOCHS:
        break

    print(f"\n{'─'*65}")
    print(f"▶  {phase['name']}  |  LR={phase['lr']:.0e}  |  up to {phase['epochs']} ep")
    print(f"   Total so far: {total_epochs_run} ep  |  "
          f"Best AUC={best_val_auc:.4f}  Acc={best_val_acc:.4f}")

    set_unfreeze(backbone, phase['pct'])
    compile_model(model, phase['lr'], wd=phase['wd'])

    run_eps = min(phase['epochs'], MAX_TOTAL_EPOCHS - total_epochs_run)

    def make_lr_fn(lr, warmup, total):
        def schedule(epoch):
            return warmup_cosine(epoch, total, warmup, lr)
        return schedule

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f'{OUTPUT_DIR}/best_overall.keras',
            monitor='val_auc', mode='max',
            save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=14,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.CSVLogger(
            f'{OUTPUT_DIR}/phase{ph_i+1}_log.csv'
        ),
        tf.keras.callbacks.LearningRateScheduler(
            make_lr_fn(phase['lr'], phase['warmup'], run_eps),
            verbose=0
        ),
    ]

    if phase['swa']:
        swa_cb = SWACallback(
            start_epoch=3,
            save_path=f'{OUTPUT_DIR}/best_swa.keras'
        )
        callbacks.append(swa_cb)
        print(f"   SWA: active from epoch 3")

    h = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=run_eps,
        class_weight=class_weights_tuned,
        callbacks=callbacks,
        verbose=1
    )

    hist_auc    = h.history.get('val_auc',      [0])
    hist_acc    = h.history.get('val_accuracy',  [0])
    ph_best_auc = max(hist_auc)
    ph_best_acc = max(hist_acc)
    eps_this_ph = len(hist_auc)
    total_epochs_run += eps_this_ph

    if ph_best_auc > best_val_auc: best_val_auc = ph_best_auc
    if ph_best_acc > best_val_acc: best_val_acc = ph_best_acc

    all_phase_log.append({
        'phase': phase['name'], 'epochs': eps_this_ph,
        'total': total_epochs_run,
        'val_auc': round(ph_best_auc, 4),
        'val_acc': round(ph_best_acc, 4),
    })

    print(f"\n   {phase['name']} DONE:")
    print(f"   val_auc={ph_best_auc:.4f} {'✅' if ph_best_auc>=TARGET_AUC else ''}")
    print(f"   val_acc={ph_best_acc:.4f} {'✅' if ph_best_acc>=TARGET_ACC else ''}")

    if best_val_auc >= TARGET_AUC and best_val_acc >= TARGET_ACC:
        targets_met = True
        print(f"\n{'='*65}")
        print(f"🎯🎯🎯  BOTH TARGETS MET — {total_epochs_run} epochs total!")
        print(f"{'='*65}")
        break

print(f"\n{'='*65}")
print(f"AUTOLOOP COMPLETE")
print(f"val_auc = {best_val_auc:.4f}  {'✅ TARGET MET' if best_val_auc>=TARGET_AUC else '❌ below target'}")
print(f"val_acc = {best_val_acc:.4f}  {'✅ TARGET MET' if best_val_acc>=TARGET_ACC else '❌ below target'}")
print(f"Total epochs: {total_epochs_run}")
print(f"\nPhase breakdown:")
for row in all_phase_log:
    flag = "✅" if (row['val_auc']>=TARGET_AUC and row['val_acc']>=TARGET_ACC) else "  "
    print(f"  {flag} {row['phase']:42s} | ep={row['epochs']:2d} | "
          f"AUC={row['val_auc']:.4f} | Acc={row['val_acc']:.4f}")



In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 12 (FIXED): TTA + THRESHOLD OPTIMIZATION — memory-safe version
# ════════════════════════════════════════════════════════════════════════

import gc
gc.collect()
tf.keras.backend.clear_session()

# ── Load model (weights-only to avoid FrozenBNBackbone error) ─────────
best_model, _ = build_model()
best_model.load_weights(f'{OUTPUT_DIR}/best_overall.keras')
print("✅ Model loaded")

# ── TTA config — REDUCED to prevent OOM ──────────────────────────────
N_TTA        = 4      # reduced from 8 → 4 (still great, half the memory)
TTA_BATCH    = 4      # only 4 images per batch (was 24) — key fix
TTA_AUGS_USE = TTA_AUGS[:N_TTA]

def tta_predict_val_safe(model, val_df, meta_cols, n_tta=4, batch_size=4):
    """Memory-safe TTA: tiny batches, explicit cleanup between passes."""
    paths  = val_df['image_path'].values
    metas  = val_df[meta_cols].values.astype(np.float32)
    labels = val_df['label'].values
    n      = len(paths)

    all_probs = np.zeros((n, NUM_CLASSES), dtype=np.float32)

    for tta_i, aug in enumerate(TTA_AUGS_USE):
        print(f"  TTA pass {tta_i+1}/{n_tta}...", end=' ')
        pass_probs = []

        for start in range(0, n, batch_size):
            end  = min(start + batch_size, n)
            imgs = []
            for j in range(start, end):
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, IMAGE_SIZE).astype(np.uint8)
                img = aug(image=img)['image'].astype(np.float32)
                imgs.append(img)

            probs = model.predict(
                {'imageinput':    np.array(imgs, dtype=np.float32),
                 'metadatainput': metas[start:end]},
                verbose=0
            )
            pass_probs.append(probs)

            # Explicit cleanup every 100 batches
            if (start // batch_size) % 100 == 0:
                gc.collect()

        all_probs += np.concatenate(pass_probs, axis=0)
        print(f"done")
        gc.collect()

    all_probs /= n_tta
    return all_probs, labels

print(f"\n🔮 Running {N_TTA}-pass TTA on {len(val_df):,} images (batch={TTA_BATCH})...")
print(f"   Expected time: ~10-20 min\n")

val_probs_tta, val_true = tta_predict_val_safe(
    best_model, val_df, META_COLS, n_tta=N_TTA, batch_size=TTA_BATCH
)
print("✅ TTA complete")

# ── Threshold optimization ────────────────────────────────────────────
from sklearn.metrics import f1_score, accuracy_score

print("\n🎯 Optimizing per-class thresholds...")

best_thresholds = [1/NUM_CLASSES] * NUM_CLASSES
best_f1 = 0.0

# Grid search over melanoma threshold (most important)
for t_mel in np.arange(0.15, 0.55, 0.02):
    for t_ben in np.arange(0.25, 0.65, 0.02):
        t_other = 1.0 - t_mel - t_ben
        if t_other < 0.05 or t_other > 0.70:
            continue
        thresholds = [t_mel, t_ben, t_other]
        
        # Apply thresholds: divide each prob by its threshold, pick argmax
        scaled = val_probs_tta / np.array(thresholds)
        preds  = np.argmax(scaled, axis=1)
        f1     = f1_score(val_true, preds, average='macro')
        
        if f1 > best_f1:
            best_f1        = f1
            best_thresholds = thresholds

print(f"✅ Best thresholds: Mel={best_thresholds[0]:.3f}  "
      f"Ben={best_thresholds[1]:.3f}  "
      f"OthMal={best_thresholds[2]:.3f}")

# ── Final accuracy with optimized thresholds ─────────────────────────
scaled_final = val_probs_tta / np.array(best_thresholds)
final_preds  = np.argmax(scaled_final, axis=1)
raw_preds    = np.argmax(val_probs_tta, axis=1)

acc_raw  = accuracy_score(val_true, raw_preds)
acc_opt  = accuracy_score(val_true, final_preds)
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(val_true, classes=[0,1,2])
auc_val = roc_auc_score(y_bin, val_probs_tta, multi_class='ovr', average='macro')

print(f"\n{'='*50}")
print(f"RESULTS AFTER TTA + THRESHOLD OPTIMIZATION")
print(f"{'='*50}")
print(f"AUC              : {auc_val:.4f}")
print(f"Accuracy (raw)   : {acc_raw:.4f}  ({acc_raw*100:.1f}%)")
print(f"Accuracy (tuned) : {acc_opt:.4f}  ({acc_opt*100:.1f}%)")
print(f"Macro F1 (tuned) : {best_f1:.4f}")
print(f"\nGain from threshold opt: +{(acc_opt-acc_raw)*100:.1f}%")
print(f"{'='*50}")

# Save thresholds for use in test cells
np.save(f'{OUTPUT_DIR}/best_thresholds.npy', np.array(best_thresholds))
print(f"\n💾 Thresholds saved → {OUTPUT_DIR}/best_thresholds.npy")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 13 PREAMBLE: run this if you restarted the kernel
# ════════════════════════════════════════════════════════════════════════

import os, gc
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2

# ── Re-run these minimal cells first ─────────────────────────────────
# You MUST rerun Cell 1, Cell 2 before this
# (so OUTPUT_DIR, IMAGE_SIZE, CLASS_NAMES, META_COLS etc. exist)
# Then run Cell 3→6 quickly to rebuild val_df and META_COLS

# ── Load saved thresholds ─────────────────────────────────────────────
best_thresholds = np.load(f'{OUTPUT_DIR}/best_thresholds.npy').tolist()
print(f"✅ Thresholds loaded: {[f'{t:.3f}' for t in best_thresholds]}")

# ── Reload model (weights method to avoid FrozenBNBackbone error) ─────
best_model, _ = build_model()
best_model.load_weights(f'{OUTPUT_DIR}/best_overall.keras')
print("✅ Model loaded — ready for Cell 13")

In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 13: TFLite EXPORT FOR Pi5 AI HAT  ← Run AFTER training finishes
# ════════════════════════════════════════════════════════════════════════

def representative_dataset():
    for (inputs, _) in val_ds.take(100):
        yield [inputs['imageinput'], inputs['metadatainput']]

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations             = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS
]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
pi_path = f'{OUTPUT_DIR}/skin_cancer_pi5_v3.tflite'
with open(pi_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(pi_path) / 1e6
print(f"✅ Pi5 TFLite model: {size_mb:.1f}MB (INT8)")
print(f"   Estimated inference on Pi5 AI HAT: ~45ms/image")
print(f"\n📁 All outputs in {OUTPUT_DIR}/:")
print(f"   best_overall.keras           — best checkpoint")
print(f"   best_swa.keras               — SWA averaged model")
print(f"   skin_cancer_pi5_v3.tflite    — INT8 for Pi5 AI HAT")
print(f"   inference_thresholds.json    — load on Pi5 for max accuracy")
print(f"   phase1_log.csv ... phaseN_log.csv — training history")










In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 14A: BATCH AUTO-TEST — randomly samples images and shows results
# Run this AFTER Cell 11 (training). No inputs needed.
# ════════════════════════════════════════════════════════════════════════

import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import cv2
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Replace lines 15-22 in Cell 14A with this:
best_model, _ = build_model()
best_model.load_weights(f'{OUTPUT_DIR}/best_overall.keras')
print("✅ Model loaded")

# ── Config ────────────────────────────────────────────────────────────
N_TEST        = 36       # number of images to test (must be perfect square or close)
N_COLS        = 6        # images per row
USE_TTA       = False    # set True for better accuracy (slower — 8 passes per image)
SAMPLE_SPLIT  = 'val'    # 'val' or 'train'

source_df = val_df if SAMPLE_SPLIT == 'val' else train_df

# ── Sample images (balanced across classes) ───────────────────────────
per_class = N_TEST // NUM_CLASSES
sampled_rows = []
for cls in range(NUM_CLASSES):
    cls_df = source_df[source_df['label'] == cls]
    sampled_rows.append(cls_df.sample(min(per_class, len(cls_df)), random_state=None))
sample_df = pd.concat(sampled_rows).sample(frac=1, random_state=None).reset_index(drop=True)

# ── Run predictions ───────────────────────────────────────────────────
def predict_single(model, img_path, meta_row, use_tta=False):
    """Run prediction on one image, optionally with TTA."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE).astype(np.float32)
    meta = np.array(meta_row, dtype=np.float32)[np.newaxis, :]

    if use_tta:
        all_probs = []
        for aug in TTA_AUGS:
            aug_img = aug(image=img.astype(np.uint8))['image'].astype(np.float32)
            probs = model.predict(
                {'imageinput':    aug_img[np.newaxis],
                 'metadatainput': meta},
                verbose=0
            )
            all_probs.append(probs[0])
        return np.mean(all_probs, axis=0)
    else:
        probs = model.predict(
            {'imageinput':    img[np.newaxis],
             'metadatainput': meta},
            verbose=0
        )
        return probs[0]

print(f"\n🔮 Running predictions on {len(sample_df)} images{'  (TTA)' if USE_TTA else ''}...")
all_probs  = []
all_preds  = []
all_true   = []
all_imgs   = []

for i, row in sample_df.iterrows():
    img_path  = row['image_path']
    meta_vals = row[META_COLS].values
    true_lbl  = int(row['label'])

    probs  = predict_single(best_model, img_path, meta_vals, use_tta=USE_TTA)
    pred   = np.argmax(probs)

    # Load display image
    disp_img = cv2.imread(img_path)
    disp_img = cv2.cvtColor(disp_img, cv2.COLOR_BGR2RGB)
    disp_img = cv2.resize(disp_img, (224, 224))

    all_probs.append(probs)
    all_preds.append(pred)
    all_true.append(true_lbl)
    all_imgs.append(disp_img)

    if (len(all_preds)) % 6 == 0:
        print(f"   {len(all_preds)}/{len(sample_df)} done...")

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)
acc = np.mean(all_preds == all_true)
print(f"\n✅ Done. Accuracy on this sample: {acc:.1%}")

# ── Plot image grid ───────────────────────────────────────────────────
n_rows  = int(np.ceil(len(sample_df) / N_COLS))
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3.2, n_rows * 3.6))
axes = axes.flatten()

LABEL_SHORT  = ['Mel', 'Benign', 'OtherMal']
LABEL_COLORS = ['#E74C3C', '#2ECC71', '#3498DB']

for idx in range(len(sample_df)):
    ax     = axes[idx]
    img    = all_imgs[idx]
    true   = all_true[idx]
    pred   = all_preds[idx]
    probs  = all_probs[idx]
    conf   = probs[pred] * 100

    correct = (pred == true)
    border_color = '#00C851' if correct else '#FF4444'

    # Add colored border
    bordered = np.pad(img, ((4, 4), (4, 4), (0, 0)), mode='constant',
                      constant_values=0)
    bordered[:4,  :, :] = [int(c * 255) for c in
                             plt.cm.colors.to_rgb(border_color)]
    bordered[-4:, :, :] = [int(c * 255) for c in
                             plt.cm.colors.to_rgb(border_color)]
    bordered[:, :4,  :] = [int(c * 255) for c in
                             plt.cm.colors.to_rgb(border_color)]
    bordered[:, -4:, :] = [int(c * 255) for c in
                             plt.cm.colors.to_rgb(border_color)]

    ax.imshow(bordered)
    ax.axis('off')

    icon = '✓' if correct else '✗'
    title = (f"{icon} True: {CLASS_NAMES[true]}\n"
             f"Pred: {CLASS_NAMES[pred]} ({conf:.0f}%)")
    title_color = '#00C851' if correct else '#FF4444'
    ax.set_title(title, fontsize=7.5, color=title_color,
                 fontweight='bold', pad=2)

    # Mini probability bars
    bar_ax = ax.inset_axes([0, -0.35, 1, 0.28])
    bars = bar_ax.barh(range(NUM_CLASSES), probs * 100,
                       color=LABEL_COLORS, height=0.6)
    bar_ax.set_xlim(0, 100)
    bar_ax.set_yticks(range(NUM_CLASSES))
    bar_ax.set_yticklabels(LABEL_SHORT, fontsize=6)
    bar_ax.tick_params(axis='x', labelsize=5)
    bar_ax.axvline(50, color='gray', lw=0.5, ls='--')
    bar_ax.set_xlabel('%', fontsize=5)

for idx in range(len(sample_df), len(axes)):
    axes[idx].axis('off')

fig.suptitle(
    f'Batch Test — {SAMPLE_SPLIT} set  |  '
    f'Accuracy: {acc:.1%}  |  '
    f'{"TTA (8-pass)" if USE_TTA else "Single pass"}',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout(h_pad=2.5)
plt.savefig(f'{OUTPUT_DIR}/batch_test_results.png', dpi=120,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"📁 Saved → {OUTPUT_DIR}/batch_test_results.png")

# ── Confusion matrix ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title('Confusion Matrix (normalized)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/batch_test_confusion.png', dpi=120,
            bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n📊 Classification Report:")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=3))


In [ ]:

# ════════════════════════════════════════════════════════════════════════
# CELL 14B: INTERACTIVE SINGLE-IMAGE PREDICTOR
# ════════════════════════════════════════════════════════════════════════
# ── INPUT: fill in these fields ───────────────────────────────────────

IMAGE_PATH = 'ISIC-images/ISIC_0024306.jpg'   # ← change to any image path

# Patient metadata — fill in what you know, leave others as None
AGE         = 58          # patient age in years  (None = use median ~58)
SEX         = 'male'      # 'male', 'female', or None
BODY_SITE   = 'lower extremity'   # options below ↓ or None

# Valid BODY_SITE options:
#   'anterior torso', 'head/neck', 'lateral torso',
#   'lower extremity', 'oral/genital', 'palms/soles',
#   'posterior torso', 'upper extremity'
#   (set to None if unknown)

LESION_SIZE_MM    = None  # longest diameter in mm (None = use median ~10mm)
FITZPATRICK_TYPE  = None  # skin type 1–6 (None = use median = 3)
MELANOCYTIC       = None  # True if lesion is melanocytic, False if not, None if unknown
USE_TTA_SINGLE    = True  # True = 8-pass TTA (more accurate, ~1-2 sec slower)

# ── ────────────────────────────────────────────────────────────────────
# DON'T TOUCH BELOW — just run the cell after filling in the above

def build_meta_for_prediction(age, sex, body_site, lesion_mm,
                               fitzpatrick, melanocytic, meta_cols):
    """Construct the metadata vector matching META_COLS."""
    row = {}

    # Age
    age_val = age if age is not None else 58.0
    row['age_normalized'] = age_val / 100.0
    row['age_norm']       = age_val / 100.0
    row['age_lt35']       = float(age_val < 35)
    row['age_35_50']      = float(35 <= age_val < 50)
    row['age_50_65']      = float(50 <= age_val < 65)
    row['age_gt65']       = float(age_val >= 65)

    # Sex
    sex_map = {'male': 0.0, 'female': 1.0, None: 0.5}
    row['sex_encoded'] = sex_map.get(sex, 0.5)
    row['sex_enc']     = row['sex_encoded']

    # Body site one-hot columns
    all_site_cols = [c for c in meta_cols if c.startswith('site_')]
    for sc in all_site_cols:
        # e.g. 'site_lower extremity' matches body_site 'lower extremity'
        site_name = sc.replace('site_', '')
        row[sc] = 1.0 if (body_site and site_name == body_site) else 0.0

    # Lesion size
    size_val = lesion_mm if lesion_mm is not None else 10.0
    row['size_norm'] = size_val / 50.0

    # Fitzpatrick
    fitz_val = fitzpatrick if fitzpatrick is not None else 3
    row['skin_type_norm'] = fitz_val / 6.0

    # Melanocytic
    if melanocytic is True:
        row['melanocytic_flag'] = 1.0
    elif melanocytic is False:
        row['melanocytic_flag'] = 0.0
    else:
        row['melanocytic_flag'] = 0.5

    # Build vector in correct META_COLS order
    vec = np.array([row.get(col, 0.0) for col in meta_cols], dtype=np.float32)
    return vec

# ── Build metadata vector ─────────────────────────────────────────────
meta_vec = build_meta_for_prediction(
    AGE, SEX, BODY_SITE, LESION_SIZE_MM,
    FITZPATRICK_TYPE, MELANOCYTIC, META_COLS
)

# ── Load and preprocess image ─────────────────────────────────────────
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"❌ Image not found: {IMAGE_PATH}")

img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_disp = cv2.resize(img_rgb, (380, 380))
img_f32  = img_disp.astype(np.float32)

# ── Run prediction ────────────────────────────────────────────────────
if USE_TTA_SINGLE:
    all_probs = []
    for aug in TTA_AUGS:
        aug_img = aug(image=img_disp)['image'].astype(np.float32)
        p = best_model.predict(
            {'imageinput':    aug_img[np.newaxis],
             'metadatainput': meta_vec[np.newaxis]},
            verbose=0
        )
        all_probs.append(p[0])
    probs = np.mean(all_probs, axis=0)
else:
    probs = best_model.predict(
        {'imageinput':    img_f32[np.newaxis],
         'metadatainput': meta_vec[np.newaxis]},
        verbose=0
    )[0]

pred_class = np.argmax(probs)
confidence = probs[pred_class] * 100

# ── Risk level ────────────────────────────────────────────────────────
mel_prob  = probs[0] * 100   # Melanoma probability
other_prob = probs[2] * 100  # Other malignant probability

if mel_prob >= 40 or other_prob >= 40:
    risk_level = '🔴 HIGH RISK — Refer urgently'
    risk_color = '#FF4444'
elif mel_prob >= 20 or other_prob >= 25:
    risk_level = '🟡 MODERATE RISK — Clinical review recommended'
    risk_color = '#FFA500'
else:
    risk_level = '🟢 LOWER RISK — Likely benign'
    risk_color = '#00C851'

# ── Display ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5.5))
gs  = fig.add_gridspec(1, 3, width_ratios=[1.2, 1.2, 1.0], wspace=0.35)

# Panel 1: Image
ax_img = fig.add_subplot(gs[0])
ax_img.imshow(img_disp)
ax_img.set_title('Input Image', fontsize=12, fontweight='bold')
ax_img.axis('off')
# Add annotation
fname = os.path.basename(IMAGE_PATH)
ax_img.text(0.5, -0.04, fname, transform=ax_img.transAxes,
            ha='center', fontsize=8, color='gray')

# Panel 2: Probability bar chart
ax_bar = fig.add_subplot(gs[1])
colors_bar = LABEL_COLORS.copy()
colors_bar[pred_class] = '#FFD700'  # Highlight predicted class

bars = ax_bar.barh(CLASS_NAMES[::-1], probs[::-1] * 100,
                   color=colors_bar[::-1], height=0.55,
                   edgecolor='white', linewidth=1.5)
ax_bar.set_xlim(0, 100)
ax_bar.axvline(50, color='gray', lw=1, ls='--', alpha=0.5)
ax_bar.set_xlabel('Probability (%)', fontsize=10)
ax_bar.set_title('Class Probabilities', fontsize=12, fontweight='bold')
ax_bar.tick_params(axis='y', labelsize=10)

# Add percentage labels on bars
for bar, prob in zip(bars, probs[::-1]):
    ax_bar.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
                f'{prob*100:.1f}%', va='center', fontsize=10, fontweight='bold')

# Highlight winning bar
bars[NUM_CLASSES - 1 - pred_class].set_edgecolor('#FFD700')
bars[NUM_CLASSES - 1 - pred_class].set_linewidth(3)

ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

# Panel 3: Summary
ax_sum = fig.add_subplot(gs[2])
ax_sum.axis('off')

summary_text = [
    ('PREDICTION', f'{CLASS_NAMES[pred_class]}', 14, '#333333', 'bold'),
    ('Confidence', f'{confidence:.1f}%', 12, LABEL_COLORS[pred_class], 'bold'),
    ('', '', 10, 'white', 'normal'),
    ('RISK ASSESSMENT', '', 9, '#555555', 'bold'),
    (risk_level, '', 9, risk_color, 'bold'),
    ('', '', 8, 'white', 'normal'),
    ('Patient Info', '', 8, '#555555', 'bold'),
    (f"Age: {AGE if AGE else 'unknown'}", '', 8, '#333333', 'normal'),
    (f"Sex: {SEX if SEX else 'unknown'}", '', 8, '#333333', 'normal'),
    (f"Site: {BODY_SITE if BODY_SITE else 'unknown'}", '', 8, '#333333', 'normal'),
    (f"Lesion Ø: {f'{LESION_SIZE_MM}mm' if LESION_SIZE_MM else 'unknown'}", '', 8, '#333333', 'normal'),
    (f"Skin type: {FITZPATRICK_TYPE if FITZPATRICK_TYPE else 'unknown'}", '', 8, '#333333', 'normal'),
    ('', '', 8, 'white', 'normal'),
    (f"TTA: {'8-pass' if USE_TTA_SINGLE else 'single'}", '', 8, '#888888', 'normal'),
]

y = 0.97
for text, subtext, fsize, color, weight in summary_text:
    ax_sum.text(0.05, y, text, transform=ax_sum.transAxes,
                fontsize=fsize, color=color, fontweight=weight, va='top')
    y -= 0.075

# Risk badge background
ax_sum.add_patch(plt.Rectangle(
    (0.0, 0.55), 1.0, 0.13,
    transform=ax_sum.transAxes,
    facecolor=risk_color, alpha=0.12, zorder=0
))

fig.suptitle(
    f'⚠️  RESEARCH TOOL ONLY — Not for clinical diagnosis',
    fontsize=9, color='#888888', y=0.01
)

plt.savefig(f'{OUTPUT_DIR}/single_prediction.png', dpi=130,
            bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n{'='*55}")
print(f"PREDICTION:  {CLASS_NAMES[pred_class]}  ({confidence:.1f}% confidence)")
print(f"RISK:        {risk_level}")
print(f"\nPer-class probabilities:")
for i, name in enumerate(CLASS_NAMES):
    bar = '█' * int(probs[i] * 30)
    print(f"  {name:18s}: {probs[i]*100:5.1f}%  {bar}")
print(f"\n📁 Saved → {OUTPUT_DIR}/single_prediction.png")
print(f"\n⚠️  DISCLAIMER: Research prototype only. Not validated for clinical use.")